In [237]:
from generate_utils import load_GraphModel, load_BiLSTMModel, load_TokenBiLSTMModel, load_LoRASEModel, load_AdapterModel
import torch
import numpy as np
import pickle
from GridMLM_tokenizers import CSGridMLMTokenizer
import os
from dotenv import load_dotenv
from tqdm import tqdm

from eval_utils import vec_ser_evidence_for_sequence_in_file, extract_topk_sequences_from_evidence, text_topk_of_chords_string_in_file
from graph_utils import graph_from_string

from langchain_ollama import ChatOllama
from langchain.tools import tool

from ollama import chat
from ollama import ChatResponse

# Load environment variables from .env file
load_dotenv()

True

In [238]:
tokenizer = CSGridMLMTokenizer(
    fixed_length=80,
    quantization='4th',
    intertwine_bar_info=True,
    trim_start=False,
    use_pc_roll=True,
    use_full_range_melody=False
)

def absoluteFilePaths(directory):
    file_names = []
    file_paths = []
    for dirpath,_,filenames in os.walk(directory):
        for f in filenames:
            if f.endswith( ('.mid', '.midi', '.mxl', '.xml', '.musicxml') ):
                file_names.append(f)
                file_paths.append(os.path.abspath(os.path.join(dirpath, f)))
    return file_names, file_paths
# end absoluteFilePaths

hook_file_names, hook_file_paths = absoluteFilePaths(os.getenv('VAL_HOOK'))
gjt_file_names, gjt_file_paths = absoluteFilePaths(os.getenv('VAL_GJT'))

device_name = 'cuda:2'
device = torch.device(device_name)

guide_arch = 'LoRA'
contra = True

adapter_model_path = f'saved_models/{guide_arch}/adapter/adapter_model_' + contra*'contra_' + 'jnhw.pt'
graph_adapter_model_path = f'saved_models/{guide_arch}/adapter/graph_model_' + contra*'contra_' + 'jnhw.pt'
token_adapter_model_path = f'saved_models/{guide_arch}/adapter/bilstm_model_' + contra*'contra_' + 'jnhw.pt'

token_adapter_model = load_TokenBiLSTMModel(token_adapter_model_path, tokenizer, device)
graph_adapter_model = load_GraphModel(graph_adapter_model_path, device)
adapter_model = load_AdapterModel(adapter_model_path, device)

token_adapter_model.eval()
graph_adapter_model.eval()
adapter_model.eval()

GuidanceAdapter(
  (proj): Linear(in_features=1024, out_features=512, bias=True)
)

In [239]:
# in_seq = 'b_G:7_@1;C#:7_@1;C:maj_@2'
# in_seq = 'b_D:min7_@1;G:7_@1;C:maj_@2'
in_seq = 'b_D#:7_@1;D:7_@1;C#:7_@1;C:maj7_@1'

In [240]:
g = graph_from_string(in_seq)

In [241]:
g.print_info()

Number of bars: 1
Segment bar range: [0, 1)
Segment graph features:
HeteroData(
  pitch={ x=[12, 12] },
  event={
    num_nodes=4,
    x=[4, 1],
  },
  (pitch, participates, event)={
    edge_index=[2, 16],
    edge_attr=[16, 5],
  },
  (event, next, event)={
    edge_index=[2, 3],
    edge_attr=[3, 6],
  }
)
Segment graph bars:
Bar 1:
Bar token positions: [2, 2, 2, 2, 2, 2, 2, 2]
Number of chord objects in bar: 4
Chord object 1:
Chord label: D#:7
Pitch classes: [1, 3, 7, 10]
Root: 3
Chord ID: 100
Bar Positions: [0, 1]
Token Positions: [2, 2]
Graph Features:
tensor([[0., 0., 0., 1., 0.],
        [1., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0.],
        [0., 0., 1., 0., 0.]])
BiLSTM Features
tensor([0., 1., 0., 1., 0., 0., 0., 1., 0., 0., 1., 0.])
Chord object 2:
Chord label: D:7
Pitch classes: [0, 2, 6, 9]
Root: 2
Chord ID: 71
Bar Positions: [2, 3]
Token Positions: [2, 2]
Graph Features:
tensor([[0., 0., 0., 1., 0.],
        [1., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0.],
       

In [242]:
seg = g.segment_graph

pitch_list = seg["pitch"].x.tolist()
event_list = seg["event"].x.squeeze(-1).tolist()

participates_index = seg["pitch", "participates", "event"].edge_index
participates_attr = seg["pitch", "participates", "event"].edge_attr
pitch_participates = [
    (int(u), int(v), attr.tolist())
    for u, v, attr in zip(
        participates_index[0],
        participates_index[1],
        participates_attr
    )
]

next_index = seg["event", "next", "event"].edge_index
next_attr = seg["event", "next", "event"].edge_attr
event_next = [
    (int(u), int(v), attr.tolist())
    for u, v, attr in zip(
        next_index[0],
        next_index[1],
        next_attr
    )
]

# previous_root_retention,
# current_root_retention,
# common_pitch_class_ratio,
# upward_semitone_resolution_to_root,
# downward_semitone_resolution_to_root,
# descending_fifth_root_motion
transition_properties = [
    'previous root retention',
    'current root retention',
    'common pitch class ration',
    'upward semitone resolution to root',
    'downward semitone resolution to root',
    'descending fifth root motion'
]

In [243]:
# print(participates_index)
# print(participates_attr)
# print(next_index)
# print(next_attr)

print(len(pitch_participates))
print(len(event_next))

16
3


In [244]:
print(event_next[0])

(0, 1, [0.0, 0.0, 0.0, 1.0, 1.0, 0.0])


In [245]:
transition_counter = 0
transition_descriptions = []
for ne in event_next:
    transition_counter += 1
    transition_description = f'Transition {transition_counter} has: '
    transition_description += f'{ne[2][2]} of pitch classes between chords retained'
    for i, attrs in enumerate(ne[2]):
        if attrs == 1:
            transition_description += ', ' + transition_properties[i]
    transition_description += '. '
    transition_descriptions.append(transition_description)

In [246]:
for td in transition_descriptions:
    print(td)

Transition 1 has: 0.0 of pitch classes between chords retained, upward semitone resolution to root, downward semitone resolution to root. 
Transition 2 has: 0.0 of pitch classes between chords retained, upward semitone resolution to root, downward semitone resolution to root. 
Transition 3 has: 0.1428571492433548 of pitch classes between chords retained, upward semitone resolution to root, downward semitone resolution to root. 


In [247]:
y = graph_adapter_model(seg)

In [248]:
print(y)

tensor([-2.3739e-01,  1.8117e-01,  8.8747e-02,  2.7542e-01, -2.5755e-01,
        -2.7376e-01,  8.6209e-02,  2.4831e-01, -8.2213e-02, -2.6330e-01,
        -2.3921e-01, -1.3045e-02,  3.2416e-01,  1.9970e-01,  1.7948e-01,
        -2.3956e-02, -6.8115e-02, -3.7143e-02, -2.0507e-01, -1.5693e-01,
         3.5116e-02,  6.1551e-02,  2.3594e-01, -1.3815e-02,  3.1802e-01,
         1.9607e-01, -3.2423e-01,  4.3358e-01,  3.0546e-01,  1.4036e-01,
        -1.1516e-01,  1.3235e-01, -1.9533e-01,  1.7004e-01,  2.4015e-01,
         6.7490e-02, -4.0673e-01, -1.9524e-01,  2.6998e-01, -1.3838e-01,
         2.1887e-01,  2.4217e-01, -2.1338e-01, -1.0564e-01, -2.4640e-02,
         2.0923e-01, -1.9718e-01,  1.6707e-01, -7.9788e-02,  1.5660e-02,
        -1.1252e-01, -5.9694e-02,  9.5257e-02, -2.0019e-01,  1.2583e-01,
        -2.6417e-01,  2.9413e-03,  3.0989e-02,  3.2652e-01, -6.7419e-02,
        -5.3582e-02, -4.5413e-02, -4.2516e-02, -6.6821e-02,  2.3579e-01,
         2.4520e-02,  1.3386e-01,  6.2837e-02, -1.1

In [249]:
# file_path = gjt_file_paths[3]
file_path = gjt_file_paths[0] # query: ['D#:7', 'D:7', 'C#:7', 'C:maj7'] | found in bars 7 - 8: ['A:7', 'D:7', 'G:maj6', 'C:maj6'],

In [250]:
bars_string, evidence = vec_ser_evidence_for_sequence_in_file(
    in_seq,
    file_path,
    tokenizer,
    graph_model=graph_adapter_model,
    bilstm_model=None,
    token_model=token_adapter_model,
    adapter_model=adapter_model,
    max_seq_len=16
)

In [251]:
print(evidence[1]['graph'])
print(evidence[2]['graph'])
print(evidence[3]['graph'])

[0.11593911 0.37166706 0.17667924 0.22949851 0.03563061 0.37166706
 0.1068091  0.42369241 0.11499019 0.20632616 0.02681946 0.2088972
 0.02681946 0.1131231  0.03022126 0.1131231 ]
[0.29435384 0.22672351 0.18465161 0.09613281 0.16591549 0.2949321
 0.16814025 0.43187481 0.13370867 0.24266873 0.13752556 0.25870001
 0.07986606 0.1407757  0.07861579]
[ 0.2526297   0.24348021  0.06918696  0.18210037  0.20488897  0.19596642
  0.25787979  0.27967542  0.18814096  0.0429948   0.143031    0.0344744
  0.11389184 -0.00040103]


In [252]:
print(evidence[1]['token'])
print(evidence[2]['token'])
print(evidence[3]['token'])

[ 0.41946438  0.19372354  0.15762506 -0.13624662  0.1728946   0.19372354
  0.42965832  0.24699345  0.45591679  0.37611926  0.03737637  0.17540513
  0.03737637  0.25764206  0.00359569  0.25764206]
[ 0.32065076  0.16568679  0.23812976 -0.04859932  0.07553114  0.47617304
  0.20515844  0.54071236  0.23750576  0.51471436  0.07709266  0.2845484
  0.19531184  0.24694869  0.20555724]
[ 0.24123077  0.22774898  0.11299083 -0.0775756   0.3029443   0.24632415
  0.46861014  0.35094994  0.35429451  0.40434551  0.27088216  0.19144692
  0.28090456  0.14894891]


In [253]:
print(evidence[1]['adapter'])
print(evidence[2]['adapter'])
print(evidence[3]['adapter'])

[0.4764607  0.44378072 0.25058055 0.25412434 0.29800081 0.44378072
 0.48755544 0.49627221 0.52853    0.45136252 0.163607   0.34294996
 0.163607   0.35144114 0.09797184 0.35144114]
[0.3915906  0.25185183 0.28041023 0.29692939 0.1563963  0.54759586
 0.33255154 0.58225483 0.36695462 0.53570747 0.30992332 0.49530888
 0.37781346 0.27698654 0.3641392 ]
[0.31450972 0.28288251 0.25338006 0.17247732 0.4077059  0.28865966
 0.5397917  0.39275819 0.48263025 0.35196185 0.46808428 0.33107227
 0.35647237 0.1838599 ]


In [254]:
print(evidence[2]['chord_symbols'])

[['G:maj6', 'G#:dim7', 'A:min7', 'D:7'], ['A:min7', 'D:7', 'G:maj7', 'D:min7', 'G:7'], ['G:maj7', 'D:min7', 'G:7', 'C:maj7', 'F:9'], ['C:maj7', 'F:9', 'G:maj7', 'E:7'], ['G:maj7', 'E:7', 'A:min7', 'D:7'], ['A:min7', 'D:7', 'G:maj6', 'E:min7'], ['G:maj6', 'E:min7', 'A:7', 'D:7'], ['A:7', 'D:7', 'G:maj6', 'C:maj6'], ['G:maj6', 'C:maj6', 'G:maj6', 'D:min7', 'G:7'], ['G:maj6', 'D:min7', 'G:7', 'C:maj6'], ['C:maj6', 'D:9', 'G:7'], ['D:9', 'G:7', 'C:maj6'], ['C:maj6', 'F:7', 'E:7'], ['F:7', 'E:7', 'A:min7'], ['A:min7', 'F:7', 'E:7']]


In [255]:
topk_per_model = extract_topk_sequences_from_evidence(evidence, 5)

print(topk_per_model['graph'])

{'similarities': [0.4318748116493225, 0.42369240522384644, 0.3716670572757721, 0.32284015417099, 0.30733609199523926], 'chord_symbols': [['A:7', 'D:7', 'G:maj6', 'C:maj6'], ['A:7', 'D:7'], ['A:min7', 'D:7'], ['A:7', 'D:7', 'G:maj6', 'C:maj6', 'G:maj6', 'D:min7', 'G:7', 'C:maj6'], ['G:maj6', 'G#:dim7', 'A:min7', 'D:7', 'G:maj7', 'D:min7', 'G:7', 'C:maj7', 'F:9']], 'starting_bars': [7, 7, 5, 7, 0], 'bar_lengths': [2, 1, 1, 4, 4]}


In [256]:
for k, v in topk_per_model['graph'].items():
    print(k, ': ', v)

similarities :  [0.4318748116493225, 0.42369240522384644, 0.3716670572757721, 0.32284015417099, 0.30733609199523926]
chord_symbols :  [['A:7', 'D:7', 'G:maj6', 'C:maj6'], ['A:7', 'D:7'], ['A:min7', 'D:7'], ['A:7', 'D:7', 'G:maj6', 'C:maj6', 'G:maj6', 'D:min7', 'G:7', 'C:maj6'], ['G:maj6', 'G#:dim7', 'A:min7', 'D:7', 'G:maj7', 'D:min7', 'G:7', 'C:maj7', 'F:9']]
starting_bars :  [7, 7, 5, 7, 0]
bar_lengths :  [2, 1, 1, 4, 4]


In [257]:
bars_string, in_seq_list, text_descriptions = text_topk_of_chords_string_in_file(
    in_seq,
    tokenizer,
    file_path,
    graph_model=graph_adapter_model,
    bilstm_model=None,
    token_model=token_adapter_model,
    adapter_model=adapter_model,
    max_seq_len=16,
    k=5
)

In [258]:
print(in_seq_list)

['D#:7', 'D:7', 'C#:7', 'C:maj7']


In [259]:
print(bars_string)

Piece:
bar 0: G:maj6 G#:dim7 
bar 1: A:min7 D:7 
bar 2: G:maj7 D:min7 G:7 
bar 3: C:maj7 F:9 
bar 4: G:maj7 E:7 
bar 5: A:min7 D:7 
bar 6: G:maj6 E:min7 
bar 7: A:7 D:7 
bar 8: G:maj6 C:maj6 
bar 9: G:maj6 D:min7 G:7 
bar 10: C:maj6 
bar 11: D:9 G:7 
bar 12: C:maj6 
bar 13: F:7 E:7 
bar 14: A:min7 
bar 15: F:7 E:7 



In [260]:
print(text_descriptions)

{'token': ["query: ['D#:7', 'D:7', 'C#:7', 'C:maj7'] | found in bars 7 - 8: ['A:7', 'D:7', 'G:maj6', 'C:maj6'],  with similarity: 0.5407123565673828", "query: ['D#:7', 'D:7', 'C#:7', 'C:maj7'] | found in bars 9 - 12: ['G:maj6', 'D:min7', 'G:7', 'C:maj6', 'D:9', 'G:7', 'C:maj6'],  with similarity: 0.5252280235290527", "query: ['D#:7', 'D:7', 'C#:7', 'C:maj7'] | found in bars 7 - 12: ['A:7', 'D:7', 'G:maj6', 'C:maj6', 'G:maj6', 'D:min7', 'G:7', 'C:maj6', 'D:9', 'G:7', 'C:maj6'],  with similarity: 0.5231090784072876", "query: ['D#:7', 'D:7', 'C#:7', 'C:maj7'] | found in bars 7 - 14: ['A:7', 'D:7', 'G:maj6', 'C:maj6', 'G:maj6', 'D:min7', 'G:7', 'C:maj6', 'D:9', 'G:7', 'C:maj6', 'F:7', 'E:7', 'A:min7'],  with similarity: 0.5163396000862122", "query: ['D#:7', 'D:7', 'C#:7', 'C:maj7'] | found in bars 9 - 10: ['G:maj6', 'D:min7', 'G:7', 'C:maj6'],  with similarity: 0.5147143602371216"], 'graph': ["query: ['D#:7', 'D:7', 'C#:7', 'C:maj7'] | found in bars 7 - 8: ['A:7', 'D:7', 'G:maj6', 'C:maj6'

In [261]:
print(bars_string)

Piece:
bar 0: G:maj6 G#:dim7 
bar 1: A:min7 D:7 
bar 2: G:maj7 D:min7 G:7 
bar 3: C:maj7 F:9 
bar 4: G:maj7 E:7 
bar 5: A:min7 D:7 
bar 6: G:maj6 E:min7 
bar 7: A:7 D:7 
bar 8: G:maj6 C:maj6 
bar 9: G:maj6 D:min7 G:7 
bar 10: C:maj6 
bar 11: D:9 G:7 
bar 12: C:maj6 
bar 13: F:7 E:7 
bar 14: A:min7 
bar 15: F:7 E:7 



In [262]:
string_descriptions = {
    'token': '',
    'graph': '',
    'adapter': ''
}

for k , v in text_descriptions.items():
    print(k)
    for t in v:
        print(t)
        string_descriptions[k] += t + '\n'

token
query: ['D#:7', 'D:7', 'C#:7', 'C:maj7'] | found in bars 7 - 8: ['A:7', 'D:7', 'G:maj6', 'C:maj6'],  with similarity: 0.5407123565673828
query: ['D#:7', 'D:7', 'C#:7', 'C:maj7'] | found in bars 9 - 12: ['G:maj6', 'D:min7', 'G:7', 'C:maj6', 'D:9', 'G:7', 'C:maj6'],  with similarity: 0.5252280235290527
query: ['D#:7', 'D:7', 'C#:7', 'C:maj7'] | found in bars 7 - 12: ['A:7', 'D:7', 'G:maj6', 'C:maj6', 'G:maj6', 'D:min7', 'G:7', 'C:maj6', 'D:9', 'G:7', 'C:maj6'],  with similarity: 0.5231090784072876
query: ['D#:7', 'D:7', 'C#:7', 'C:maj7'] | found in bars 7 - 14: ['A:7', 'D:7', 'G:maj6', 'C:maj6', 'G:maj6', 'D:min7', 'G:7', 'C:maj6', 'D:9', 'G:7', 'C:maj6', 'F:7', 'E:7', 'A:min7'],  with similarity: 0.5163396000862122
query: ['D#:7', 'D:7', 'C#:7', 'C:maj7'] | found in bars 9 - 10: ['G:maj6', 'D:min7', 'G:7', 'C:maj6'],  with similarity: 0.5147143602371216
graph
query: ['D#:7', 'D:7', 'C#:7', 'C:maj7'] | found in bars 7 - 8: ['A:7', 'D:7', 'G:maj6', 'C:maj6'],  with similarity: 0.431

In [270]:
central_prompt = '''
You are a music harmony expert. You will be give the chord sequence of a piece, per bar.
You will also be given a query sequence of chords.
You job is to identify chord subsequences within the piece that are similar 
(not necessarily identical) to the query sequence. You need to provide details about the 
reasons why you believe these segments are similar to the query.\n\n
'''

focus_properties_prompt = f'''
You can focus your similarity criteria toward identifying similarities per transition
in the query and the piece sequences based on the following criteria between pairs
of chords in each transition:
{transition_properties}\n\n
'''

piece_and_query_prompt = f'''
Here is the piece sequence:\n
{bars_string}\n\n
Here is the query:\n
{in_seq_list}
'''

adapter_tool_prompt = '''
You can use the output of a model that assessed the similarities (maximum 1, minimum -1)
between bar segments of the pieces and query:
''' + string_descriptions['adapter']

In [268]:
print(central_prompt + focus_properties_prompt + piece_and_query_prompt )


You are a music harmony expert. You will be give the chord sequence of one, per bar.
You will also be given a query sequence of chords.
You job is to identify chord subsequences within the piece that are similar 
(not necessarily identical) to the query sequence. You need to provide details about the 
reasons why you believe these segments are similar to the query.



You can focus your similarity criteria toward identifying similarities per transition
in the query and the piece sequences based on the following criteria between pairs
of chords in each transition:
['previous root retention', 'current root retention', 'common pitch class ration', 'upward semitone resolution to root', 'downward semitone resolution to root', 'descending fifth root motion']



Here is the piece sequence:

Piece:
bar 0: G:maj6 G#:dim7 
bar 1: A:min7 D:7 
bar 2: G:maj7 D:min7 G:7 
bar 3: C:maj7 F:9 
bar 4: G:maj7 E:7 
bar 5: A:min7 D:7 
bar 6: G:maj6 E:min7 
bar 7: A:7 D:7 
bar 8: G:maj6 C:maj6 
bar 9: G:maj

In [265]:
# model_name = 'deepseek-r1:14b'
model_name = 'qwen2.5-coder:14b'

In [266]:
response: ChatResponse = chat(
  model=model_name,
  messages=[
    {
      'role': 'user',
      'content': central_prompt + focus_properties_prompt + piece_and_query_prompt,
    }
  ],
  keep_alive=0
)
basic_response = response['message']['content']
print(basic_response)

To identify chord subsequences within the piece that are similar to the query sequence ['D#:7', 'D:7', 'C#:7', 'C:maj7'], I'll analyze each transition in the piece against the criteria you've provided:

1. **Previous Root Retention**
2. **Current Root Retention**
3. **Common Pitch Class Ratio**
4. **Upward Semitone Resolution to Root**
5. **Downward Semitone Resolution to Root**
6. **Descending Fifth Root Motion**

Let's break down each transition in the query and compare it with transitions in the piece:

### Query Transitions
1. **D#:7 → D:7**
   - **Previous Root Retention**: No (D# vs D)
   - **Current Root Retention**: Yes (D)
   - **Common Pitch Class Ratio**: 6/12 (half of the pitch classes are common, since D and D# share common tones but differ in one tone)
   - **Upward Semitone Resolution to Root**: No (D# is already a semitone above D)
   - **Downward Semitone Resolution to Root**: Yes (D# is a semitone below D)
   - **Descending Fifth Root Motion**: No (There's no fifth mo

In [269]:
print(central_prompt + focus_properties_prompt + piece_and_query_prompt + adapter_tool_prompt)


You are a music harmony expert. You will be give the chord sequence of one, per bar.
You will also be given a query sequence of chords.
You job is to identify chord subsequences within the piece that are similar 
(not necessarily identical) to the query sequence. You need to provide details about the 
reasons why you believe these segments are similar to the query.



You can focus your similarity criteria toward identifying similarities per transition
in the query and the piece sequences based on the following criteria between pairs
of chords in each transition:
['previous root retention', 'current root retention', 'common pitch class ration', 'upward semitone resolution to root', 'downward semitone resolution to root', 'descending fifth root motion']



Here is the piece sequence:

Piece:
bar 0: G:maj6 G#:dim7 
bar 1: A:min7 D:7 
bar 2: G:maj7 D:min7 G:7 
bar 3: C:maj7 F:9 
bar 4: G:maj7 E:7 
bar 5: A:min7 D:7 
bar 6: G:maj6 E:min7 
bar 7: A:7 D:7 
bar 8: G:maj6 C:maj6 
bar 9: G:maj

In [267]:
response: ChatResponse = chat(
  model=model_name,
  messages=[
    {
      'role': 'user',
      'content': central_prompt + focus_properties_prompt + piece_and_query_prompt + adapter_tool_prompt,
    }
  ],
  keep_alive=0
)
basic_response = response['message']['content']
print(basic_response)

To identify chord subsequences within the piece that are similar to the query sequence, I'll analyze each segment found by the model based on the criteria you provided:

1. **Previous Root Retention**: Do the chords share the same root or is there some kind of continuity in the roots?
2. **Current Root Retention**: Does the current chord have the same root as the previous one or not?
3. **Common Pitch Class Ratio**: Are there common pitch classes (notes) between consecutive chords?
4. **Upward Semitone Resolution to Root**: Is the next chord resolving upwards by a semitone towards its root?
5. **Downward Semitone Resolution to Root**: Is the next chord resolving downwards by a semitone towards its root?
6. **Descending Fifth Root Motion**: Is there a descending motion of the fifth between the roots of consecutive chords?

Now, let's examine each identified segment:

### Segment 1: Bars 7 - 8
**Query:** `['D#:7', 'D:7', 'C#:7', 'C:maj7']`
**Piece Sequence:** `['A:7', 'D:7', 'G:maj6', 'C